# GridOpt Lab / explore the decisions

Run after `pip install -e .`. This notebook uses the production model functions and bundled public data. RTS-GMLC is a synthetic U.S. Southwest benchmark, not an actual utility dispatch record.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from gridopt.data import SystemData
from gridopt.dispatch import solve_dispatch
from gridopt.opf import solve_opf
from gridopt.expansion import build_problem, solve_benders, solve_extensive
system = SystemData.load()
system.generators.groupby('fuel')[['pmax', 'cost_per_mwh']].agg(['count', 'mean'])

## 1. Read the renewable and net-load shape

These profiles model 2020. Real-time 5-minute MW values were averaged to hourly MW before calculating forecast errors.

In [ ]:
day = system.day('2020-08-15')
day[['load_da', 'wind_da', 'pv_da', 'net_da']].plot(figsize=(11, 4), ylabel='MW')
plt.show()

## 2. Price upward forecast risk

Compare identical energy forecasts and calibration samples. Reserve exceedance is a procurement proxy, not load shedding.

In [ ]:
t = pd.Timestamp('2020-08-15 18:00')
errors = system.errors(t)
rows = []
for weight in [0, 10, 40, 80]:
    r = solve_dispatch(system.generators, system.profiles.loc[t, 'net_da'], errors, risk_weight=weight)
    rows.append({'risk_weight': weight, 'reserve_MW': r.reserve.sum(), 'reserve_cost_USD': r.reserve_cost, 'training_CVaR_MW': r.cvar_mw})
pd.DataFrame(rows)

## 3. Separate line limits and verify the answer

A 0.65 rating factor is a deliberate transmission-stress scenario. All final lines are checked.

In [ ]:
screened = solve_opf(system, t, screening=True, rating_factor=.65)
full = solve_opf(system, t, screening=False, rating_factor=.65)
print('Objective difference:', screened.objective - full.objective)
pd.DataFrame(screened.iterations)

## 4. Watch Benders close its bounds

The continuous generation investment problem uses four representative days, stylized retirement/growth and a disclosed carbon price. It is not an adequacy study or investment forecast.

In [ ]:
problem = build_problem(system, carbon_price=150)
benders = solve_benders(problem)
extensive = solve_extensive(problem)
print('Build MW [solar, wind, gas]:', benders.build)
print('Relative objective error:', abs(benders.objective-extensive.objective)/extensive.objective)
history = pd.DataFrame(benders.history)
history.plot(x='iteration', y=['lower_bound', 'upper_bound'], figsize=(10, 4), ylabel='Annualized USD')
plt.show()

## Continue with the complete experiment protocol

Run `gridopt run --output results/runs/my-run` then `gridopt report --input results/runs/my-run`. The full run includes UC, solver gaps, chronological ED holdouts and carbon sensitivity. See `docs/formulations.md` and `docs/research.md` before interpreting results.